# cuda-fused-layernorm: Colab T4 reproduction

Runs this repository's own scripts on a Colab **Tesla T4** (Runtime -> Change runtime type -> T4 GPU).
Nothing is embedded: the notebook clones the repo and runs `three_way/bench3.py`,
`three_way/bench_e2e.py` and `stability.py` exactly as committed.

Every step runs as a subprocess, so the pinned packages from `requirements-gpu.txt`
are picked up without restarting the runtime.

After the run, commit `three_way/bench3_results.json`, `three_way/bench_e2e_results.json`
and `stability_t4.txt`, save this notebook with its outputs, and update the README
numbers (then `python check_readme_numbers.py --check`).

## 0. Clone and install the pinned GPU environment

In [ ]:
REF = "master"   # branch or tag to reproduce
!git clone --depth 1 -b {REF} https://github.com/LUOaini1213/cuda-fused-layernorm
%cd cuda-fused-layernorm
!git log -1 --format='%H %cd'
!pip install -q -r requirements-gpu.txt

## 1. Hardware and versions

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv
!python -c "import torch, triton, cupy, numpy; p = torch.cuda.get_device_properties(0); print(p.name, f'sm_{p.major}{p.minor}', p.multi_processor_count, 'SM |', 'torch', torch.__version__, '| triton', triton.__version__, '| cupy', cupy.__version__, '| numpy', numpy.__version__)"

## 2. Same-GPU three-way comparison (eager / Triton / CUDA C++)

Ends with `ACCURACY GATE: PASS` or `FAIL` (exit code 1 on FAIL).

In [ ]:
%cd three_way
!python bench3.py

## 3. End-to-end Transformer forward

Global warm-up, mode order rotated each round, 3 rounds per config (median reported).

In [ ]:
!python bench_e2e.py
%cd ..

## 4. Single-pass vs two-pass stability sweep

In [ ]:
!python stability.py | tee stability_t4.txt

## 5. Collect outputs

In [ ]:
import json, os
outs = ["three_way/bench3_results.json", "three_way/bench_e2e_results.json", "stability_t4.txt"]
for f in outs:
    print(f, "OK" if os.path.exists(f) else "MISSING")
rows = json.load(open(outs[0])) if os.path.exists(outs[0]) else []
print("gate failures:", sum(len(r.get("gate_failures", [])) for r in rows))
try:
    from google.colab import files
    for f in outs:
        if os.path.exists(f):
            files.download(f)
except ImportError:
    print("Not on Colab: copy the files above manually.")